# 28. 스태킹 앙상블과 OOF 메타 학습 (Stacking)

서로 다른 기본 모델(LogisticRegression, DecisionTree, SVC)의 교차검증 예측(OOF)을 입력으로 받아 메타 모델이 최종 예측을 수행하는 `StackingClassifier`와 수동 OOF 스태킹을 구현합니다.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

# 1. StackingClassifier 활용 (5-fold OOF 예측으로 메타 모델 누수 방지)
estimators = [
    ('lr', Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=500))])),
    ('dt', DecisionTreeClassifier(random_state=42)),
    ('svc', Pipeline([('scaler', StandardScaler()), ('clf', SVC(probability=True, random_state=42))])),
]
final_estimator = LogisticRegression(max_iter=1000, solver='lbfgs')

stack_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=final_estimator,
    cv=5,
    stack_method='predict_proba',
)
stack_clf.fit(X_train, y_train)
pred = stack_clf.predict(X_test)
print(f'StackingClassifier Accuracy: {accuracy_score(y_test, pred):.4f}')
print(classification_report(y_test, pred))

In [ ]:
# 2. 수동 OOF(Out-Of-Fold) 스태킹 구현 (누수 방지)
oof_parts = []
test_parts = []
for _, model in estimators:
    oof_proba = cross_val_predict(model, X_train, y_train, cv=5, method='predict_proba')
    oof_parts.append(oof_proba)
    model.fit(X_train, y_train)
    test_parts.append(model.predict_proba(X_test))

X_meta_train = np.hstack(oof_parts)
X_meta_test = np.hstack(test_parts)

meta_model = LogisticRegression(max_iter=1000)
meta_model.fit(X_meta_train, y_train)
manual_pred = meta_model.predict(X_meta_test)
print(f'Manual OOF Stacking Accuracy: {accuracy_score(y_test, manual_pred):.4f}')